# 01 — Build Customer 360 Gold

## Purpose

Build a single analytical record for every current customer by combining customer, subscription, invoice, and payment information from the Silver layer.

The Customer 360 table will include:

- Customer profile and current status
- Subscription portfolio and contracted revenue
- Total invoiced, collected, outstanding, and voided amounts
- Payment attempts, failures, successful retries, and last payment activity
- Revenue and payment-risk indicators
- Customer-level MRR and ARR

## Sources

- `workspace.revenue_leakage_silver.customers`
- `workspace.revenue_leakage_silver.subscriptions`
- `workspace.revenue_leakage_silver.invoices`
- `workspace.revenue_leakage_silver.payments`

## Target

- `workspace.revenue_leakage_gold.customer_360`

## 1. Load and Validate Silver Sources

Load the four validated Silver tables and confirm that the expected current-state populations are available before constructing Customer 360.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SILVER_CUSTOMERS_TABLE = (
    "workspace.revenue_leakage_silver.customers"
)

SILVER_SUBSCRIPTIONS_TABLE = (
    "workspace.revenue_leakage_silver.subscriptions"
)

SILVER_INVOICES_TABLE = (
    "workspace.revenue_leakage_silver.invoices"
)

SILVER_PAYMENTS_TABLE = (
    "workspace.revenue_leakage_silver.payments"
)

GOLD_CUSTOMER_360_TABLE = (
    "workspace.revenue_leakage_gold.customer_360"
)

EXPECTED_CUSTOMER_COUNT = 5_150
EXPECTED_SUBSCRIPTION_COUNT = 6_200
EXPECTED_INVOICE_COUNT = 26_699
EXPECTED_PAYMENT_COUNT = 29_972

silver_customers_df = spark.table(
    SILVER_CUSTOMERS_TABLE
)

silver_subscriptions_df = spark.table(
    SILVER_SUBSCRIPTIONS_TABLE
)

silver_invoices_df = spark.table(
    SILVER_INVOICES_TABLE
)

silver_payments_df = spark.table(
    SILVER_PAYMENTS_TABLE
)

source_counts = {
    "customers": silver_customers_df.count(),
    "subscriptions": silver_subscriptions_df.count(),
    "invoices": silver_invoices_df.count(),
    "payments": silver_payments_df.count(),
}

for source_name, source_count in (
    source_counts.items()
):
    print(
        f"Silver {source_name}: "
        f"{source_count:,}"
    )

assert (
    source_counts["customers"]
    == EXPECTED_CUSTOMER_COUNT
), "Unexpected Silver customer count."

assert (
    source_counts["subscriptions"]
    == EXPECTED_SUBSCRIPTION_COUNT
), "Unexpected Silver subscription count."

assert (
    source_counts["invoices"]
    == EXPECTED_INVOICE_COUNT
), "Unexpected Silver invoice count."

assert (
    source_counts["payments"]
    == EXPECTED_PAYMENT_COUNT
), "Unexpected Silver payment count."

print(
    "All required Silver sources are available."
)

display(
    silver_customers_df
    .groupBy(
        "customer_status",
        "customer_segment",
    )
    .count()
    .orderBy(
        "customer_status",
        "customer_segment",
    )
)

## 2. Build Customer Subscription Metrics

Aggregate the current Silver subscription portfolio to one record per customer.

The metrics distinguish active, paused, and cancelled subscriptions. Only active subscriptions contribute to current MRR and ARR, while plan diversity and auto-renew coverage provide additional customer-value and retention context.

In [0]:
active_monthly_revenue = (
    F.when(
        F.col("subscription_status")
        == "Active",
        F.col("contracted_monthly_price"),
    )
    .otherwise(F.lit(0))
)

customer_subscription_metrics_df = (
    silver_subscriptions_df
    .groupBy("customer_id")
    .agg(
        F.count("*").alias(
            "subscription_count"
        ),
        F.sum(
            F.when(
                F.col("subscription_status")
                == "Active",
                1,
            ).otherwise(0)
        ).alias("active_subscription_count"),
        F.sum(
            F.when(
                F.col("subscription_status")
                == "Paused",
                1,
            ).otherwise(0)
        ).alias("paused_subscription_count"),
        F.sum(
            F.when(
                F.col("subscription_status")
                == "Cancelled",
                1,
            ).otherwise(0)
        ).alias(
            "cancelled_subscription_count"
        ),
        F.sum(
            F.when(
                (
                    F.col("subscription_status")
                    == "Active"
                )
                & F.col("auto_renew"),
                1,
            ).otherwise(0)
        ).alias(
            "active_auto_renew_count"
        ),
        F.round(
            F.sum(active_monthly_revenue),
            2,
        ).alias("current_mrr"),
        F.sort_array(
            F.collect_set(
                F.when(
                    F.col("subscription_status")
                    == "Active",
                    F.col("plan_name"),
                )
            )
        ).alias("active_plan_names"),
        F.min("start_date").alias(
            "first_subscription_start_date"
        ),
        F.max("start_date").alias(
            "latest_subscription_start_date"
        ),
        F.max("last_event_timestamp").alias(
            "latest_subscription_event_timestamp"
        ),
    )
    .withColumn(
        "current_arr",
        F.round(
            F.col("current_mrr") * F.lit(12),
            2,
        ),
    )
)

subscription_metric_customer_count = (
    customer_subscription_metrics_df.count()
)

duplicate_subscription_metric_customers = (
    subscription_metric_customer_count
    - customer_subscription_metrics_df
      .select("customer_id")
      .distinct()
      .count()
)

assert (
    subscription_metric_customer_count
    <= EXPECTED_CUSTOMER_COUNT
), "Subscription metrics exceed customer count."

assert (
    duplicate_subscription_metric_customers
    == 0
), "Duplicate customer subscription metrics detected."

print(
    f"Customers with subscriptions: "
    f"{subscription_metric_customer_count:,}"
)

print(
    f"Duplicate customer metrics: "
    f"{duplicate_subscription_metric_customers:,}"
)

display(
    customer_subscription_metrics_df
    .orderBy(
        F.col("current_mrr").desc(),
        "customer_id",
    )
    .limit(20)
)

## 3. Build Customer Invoice Metrics

Aggregate invoice history to one record per customer.

The metrics separate gross invoiced value, collectible revenue, collected revenue, outstanding balances, past-due exposure, open balances, and voided amounts. The latest invoice is retained to provide current billing context.

In [0]:
latest_invoice_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(
        F.col("invoice_date").desc(),
        F.col("invoice_id").desc(),
    )
)

latest_customer_invoice_df = (
    silver_invoices_df
    .withColumn(
        "_invoice_rank",
        F.row_number().over(
            latest_invoice_window
        ),
    )
    .filter(F.col("_invoice_rank") == 1)
    .select(
        "customer_id",
        F.col("invoice_id").alias(
            "latest_invoice_id"
        ),
        F.col("invoice_status").alias(
            "latest_invoice_status"
        ),
        F.col("invoice_date").alias(
            "latest_invoice_date"
        ),
        F.col("due_date").alias(
            "latest_invoice_due_date"
        ),
    )
)

customer_invoice_aggregates_df = (
    silver_invoices_df
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("invoice_count"),
        F.sum(
            F.when(
                F.col("invoice_status")
                == "Paid",
                1,
            ).otherwise(0)
        ).alias("paid_invoice_count"),
        F.sum(
            F.when(
                F.col("invoice_status")
                == "Open",
                1,
            ).otherwise(0)
        ).alias("open_invoice_count"),
        F.sum(
            F.when(
                F.col("invoice_status")
                == "Past Due",
                1,
            ).otherwise(0)
        ).alias("past_due_invoice_count"),
        F.sum(
            F.when(
                F.col("invoice_status")
                == "Voided",
                1,
            ).otherwise(0)
        ).alias("voided_invoice_count"),
        F.round(
            F.sum("invoice_total_amount"),
            2,
        ).alias("gross_invoiced_amount"),
        F.round(
            F.sum(
                F.when(
                    F.col("invoice_status")
                    != "Voided",
                    F.col("invoice_total_amount"),
                ).otherwise(F.lit(0))
            ),
            2,
        ).alias("collectible_invoice_amount"),
        F.round(
            F.sum("amount_paid"),
            2,
        ).alias("collected_amount"),
        F.round(
            F.sum("outstanding_amount"),
            2,
        ).alias("outstanding_amount"),
        F.round(
            F.sum(
                F.when(
                    F.col("invoice_status")
                    == "Past Due",
                    F.col("outstanding_amount"),
                ).otherwise(F.lit(0))
            ),
            2,
        ).alias("past_due_amount"),
        F.round(
            F.sum(
                F.when(
                    F.col("invoice_status")
                    == "Open",
                    F.col("outstanding_amount"),
                ).otherwise(F.lit(0))
            ),
            2,
        ).alias("open_outstanding_amount"),
        F.round(
            F.sum("voided_amount"),
            2,
        ).alias("voided_amount"),
        F.round(
            F.avg("invoice_total_amount"),
            2,
        ).alias("average_invoice_amount"),
        F.min("invoice_date").alias(
            "first_invoice_date"
        ),
        F.max("invoice_date").alias(
            "most_recent_invoice_date"
        ),
    )
    .withColumn(
        "collection_rate_pct",
        F.when(
            F.col("collectible_invoice_amount")
            > 0,
            F.round(
                (
                    F.col("collected_amount")
                    / F.col(
                        "collectible_invoice_amount"
                    )
                )
                * F.lit(100),
                2,
            ),
        ).otherwise(F.lit(0.00)),
    )
)

customer_invoice_metrics_df = (
    customer_invoice_aggregates_df
    .join(
        latest_customer_invoice_df,
        on="customer_id",
        how="left",
    )
)

invoice_metric_customer_count = (
    customer_invoice_metrics_df.count()
)

duplicate_invoice_metric_customers = (
    invoice_metric_customer_count
    - customer_invoice_metrics_df
      .select("customer_id")
      .distinct()
      .count()
)

assert (
    invoice_metric_customer_count
    <= EXPECTED_CUSTOMER_COUNT
), "Invoice metrics exceed customer count."

assert (
    duplicate_invoice_metric_customers
    == 0
), "Duplicate customer invoice metrics detected."

print(
    f"Customers with invoices: "
    f"{invoice_metric_customer_count:,}"
)

print(
    f"Duplicate customer metrics: "
    f"{duplicate_invoice_metric_customers:,}"
)

display(
    customer_invoice_metrics_df
    .orderBy(
        F.col("outstanding_amount").desc(),
        "customer_id",
    )
    .limit(20)
)

## 4. Build Customer Payment and Retry Metrics

Aggregate payment-attempt history to one record per customer.

The metrics distinguish first attempts from retries, measure successful recovery after failure, preserve failed and pending attempt exposure, and retain the latest payment activity for operational context.

In [0]:
latest_payment_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(
        F.col("payment_date").desc(),
        F.col("last_event_timestamp").desc(),
        F.col("attempt_number").desc(),
        F.col("payment_id").desc(),
    )
)

latest_customer_payment_df = (
    silver_payments_df
    .withColumn(
        "_payment_rank",
        F.row_number().over(
            latest_payment_window
        ),
    )
    .filter(F.col("_payment_rank") == 1)
    .select(
        "customer_id",
        F.col("payment_id").alias(
            "latest_payment_id"
        ),
        F.col("payment_status").alias(
            "latest_payment_status"
        ),
        F.col("payment_date").alias(
            "latest_payment_date"
        ),
        F.col("payment_method").alias(
            "latest_payment_method"
        ),
        F.col("payment_provider").alias(
            "latest_payment_provider"
        ),
        F.col("attempt_number").alias(
            "latest_payment_attempt_number"
        ),
    )
)

customer_payment_aggregates_df = (
    silver_payments_df
    .groupBy("customer_id")
    .agg(
        F.count("*").alias(
            "payment_attempt_count"
        ),
        F.sum(
            F.when(
                F.col("payment_status")
                == "Succeeded",
                1,
            ).otherwise(0)
        ).alias(
            "successful_payment_count"
        ),
        F.sum(
            F.when(
                F.col("payment_status")
                == "Failed",
                1,
            ).otherwise(0)
        ).alias("failed_payment_count"),
        F.sum(
            F.when(
                F.col("payment_status")
                == "Pending",
                1,
            ).otherwise(0)
        ).alias("pending_payment_count"),
        F.sum(
            F.when(
                (
                    F.col("attempt_number") == 1
                )
                & (
                    F.col("payment_status")
                    == "Succeeded"
                ),
                1,
            ).otherwise(0)
        ).alias(
            "first_attempt_success_count"
        ),
        F.sum(
            F.when(
                (
                    F.col("attempt_number") == 1
                )
                & (
                    F.col("payment_status")
                    == "Failed"
                ),
                1,
            ).otherwise(0)
        ).alias(
            "first_attempt_failure_count"
        ),
        F.sum(
            F.when(
                F.col("attempt_number") == 2,
                1,
            ).otherwise(0)
        ).alias("retry_attempt_count"),
        F.sum(
            F.when(
                (
                    F.col("attempt_number") == 2
                )
                & (
                    F.col("payment_status")
                    == "Succeeded"
                ),
                1,
            ).otherwise(0)
        ).alias(
            "successful_retry_count"
        ),
        F.sum(
            F.when(
                (
                    F.col("attempt_number") == 2
                )
                & (
                    F.col("payment_status")
                    == "Failed"
                ),
                1,
            ).otherwise(0)
        ).alias("failed_retry_count"),
        F.round(
            F.sum("transaction_amount"),
            2,
        ).alias(
            "payment_attempt_transaction_amount"
        ),
        F.round(
            F.sum("settled_amount"),
            2,
        ).alias("settled_amount"),
        F.round(
            F.sum(
                F.when(
                    (
                        F.col("attempt_number") == 2
                    )
                    & (
                        F.col("payment_status")
                        == "Succeeded"
                    ),
                    F.col("settled_amount"),
                ).otherwise(F.lit(0))
            ),
            2,
        ).alias(
            "recovered_retry_amount"
        ),
        F.round(
            F.sum(
                F.when(
                    F.col("payment_status")
                    == "Failed",
                    F.col("transaction_amount"),
                ).otherwise(F.lit(0))
            ),
            2,
        ).alias(
            "failed_attempt_transaction_amount"
        ),
        F.sort_array(
            F.collect_set(
                F.when(
                    F.col("failure_reason").isNotNull(),
                    F.col("failure_reason"),
                )
            )
        ).alias("failure_reasons"),
        F.min("payment_date").alias(
            "first_payment_date"
        ),
        F.max("payment_date").alias(
            "most_recent_payment_date"
        ),
    )
    .withColumn(
        "payment_attempt_success_rate_pct",
        F.when(
            F.col("payment_attempt_count") > 0,
            F.round(
                (
                    F.col(
                        "successful_payment_count"
                    )
                    / F.col(
                        "payment_attempt_count"
                    )
                )
                * F.lit(100),
                2,
            ),
        ).otherwise(F.lit(0.00)),
    )
    .withColumn(
        "retry_recovery_rate_pct",
        F.when(
            F.col("retry_attempt_count") > 0,
            F.round(
                (
                    F.col(
                        "successful_retry_count"
                    )
                    / F.col(
                        "retry_attempt_count"
                    )
                )
                * F.lit(100),
                2,
            ),
        ).otherwise(F.lit(0.00)),
    )
)

customer_payment_metrics_df = (
    customer_payment_aggregates_df
    .join(
        latest_customer_payment_df,
        on="customer_id",
        how="left",
    )
)

payment_metric_customer_count = (
    customer_payment_metrics_df.count()
)

duplicate_payment_metric_customers = (
    payment_metric_customer_count
    - customer_payment_metrics_df
      .select("customer_id")
      .distinct()
      .count()
)

assert (
    payment_metric_customer_count
    <= EXPECTED_CUSTOMER_COUNT
), "Payment metrics exceed customer count."

assert (
    duplicate_payment_metric_customers
    == 0
), "Duplicate customer payment metrics detected."

print(
    f"Customers with payment attempts: "
    f"{payment_metric_customer_count:,}"
)

print(
    f"Duplicate customer metrics: "
    f"{duplicate_payment_metric_customers:,}"
)

display(
    customer_payment_metrics_df
    .orderBy(
        F.col("failed_payment_count").desc(),
        F.col(
            "failed_attempt_transaction_amount"
        ).desc(),
        "customer_id",
    )
    .limit(20)
)

## 5. Assemble the Customer 360 Dataset

Join the customer profile with subscription, invoice, and payment metrics using the customer business key.

All customers remain in the dataset through left joins, including customers without invoices or payments. Missing numeric metrics are converted to zero, while unavailable activity details remain null.

A transparent rules-based score classifies customer revenue risk using past-due balances, failed retries, pending payments, paused subscriptions, customer status, and active-subscription coverage.

In [0]:
# Determine the analytics snapshot date from the latest
# customer business event.

analytics_snapshot_date = (
    silver_customers_df
    .agg(
        F.max(
            F.to_date("last_event_timestamp")
        ).alias("analytics_snapshot_date")
    )
    .first()["analytics_snapshot_date"]
)

assert analytics_snapshot_date is not None, (
    "Analytics snapshot date could not be determined."
)


# Build the base customer profile.

customer_profile_df = (
    silver_customers_df
    .select(
        "customer_id",
        "first_name",
        "last_name",
        "email",
        "country",
        "region",
        "customer_segment",
        "signup_date",
        "customer_status",
        F.col("last_event_timestamp").alias(
            "customer_last_event_timestamp"
        ),
    )
    .withColumn(
    "customer_name",
    F.concat_ws(
        " ",
        F.col("first_name"),
        F.col("last_name"),
    ),
)
)


# Join subscription, invoice, and payment metrics.

customer_360_joined_df = (
    customer_profile_df
    .join(
        customer_subscription_metrics_df,
        on="customer_id",
        how="left",
    )
    .join(
        customer_invoice_metrics_df,
        on="customer_id",
        how="left",
    )
    .join(
        customer_payment_metrics_df,
        on="customer_id",
        how="left",
    )
)


# Replace missing numeric metrics with zero.

numeric_metric_columns = [
    "subscription_count",
    "active_subscription_count",
    "paused_subscription_count",
    "cancelled_subscription_count",
    "active_auto_renew_count",
    "current_mrr",
    "current_arr",
    "invoice_count",
    "paid_invoice_count",
    "open_invoice_count",
    "past_due_invoice_count",
    "voided_invoice_count",
    "gross_invoiced_amount",
    "collectible_invoice_amount",
    "collected_amount",
    "outstanding_amount",
    "past_due_amount",
    "open_outstanding_amount",
    "voided_amount",
    "average_invoice_amount",
    "collection_rate_pct",
    "payment_attempt_count",
    "successful_payment_count",
    "failed_payment_count",
    "pending_payment_count",
    "first_attempt_success_count",
    "first_attempt_failure_count",
    "retry_attempt_count",
    "successful_retry_count",
    "failed_retry_count",
    "payment_attempt_transaction_amount",
    "settled_amount",
    "recovered_retry_amount",
    "failed_attempt_transaction_amount",
    "payment_success_rate_pct",
    "retry_recovery_rate_pct",
]

available_numeric_metric_columns = [
    column_name
    for column_name in numeric_metric_columns
    if column_name in customer_360_joined_df.columns
]

customer_360_filled_df = (
    customer_360_joined_df
    .fillna(
        0,
        subset=available_numeric_metric_columns,
    )
)


# Replace missing array metrics with empty arrays.

array_metric_columns = [
    "active_plan_names",
    "failure_reasons",
]

for column_name in array_metric_columns:
    if column_name in customer_360_filled_df.columns:
        customer_360_filled_df = (
            customer_360_filled_df
            .withColumn(
                column_name,
                F.when(
                    F.col(column_name).isNull(),
                    F.expr(
                        "CAST(array() AS ARRAY<STRING>)"
                    ),
                ).otherwise(
                    F.col(column_name)
                ),
            )
        )


# Add snapshot, tenure, and revenue-risk metrics.

customer_360_enriched_df = (
    customer_360_filled_df
    .withColumn(
        "analytics_snapshot_date",
        F.lit(analytics_snapshot_date).cast("date"),
    )
    .withColumn(
        "customer_tenure_days",
        F.datediff(
            F.col("analytics_snapshot_date"),
            F.col("signup_date"),
        ),
    )
    .withColumn(
        "revenue_at_risk_amount",
        F.round(
            F.col("outstanding_amount"),
            2,
        ),
    )
    .withColumn(
        "confirmed_leakage_amount",
        F.round(
            F.col("past_due_amount"),
            2,
        ),
    )
)


# Calculate the customer risk score.

customer_360_scored_df = (
    customer_360_enriched_df
    .withColumn(
        "risk_score",
        F.least(
            F.lit(100),
            (
                F.when(
                    F.col("customer_status") == "Inactive",
                    25,
                ).otherwise(0)
                +
                F.when(
                    F.col("past_due_amount") > 0,
                    40,
                ).otherwise(0)
                +
                F.when(
                    F.col("failed_retry_count") > 0,
                    20,
                ).otherwise(0)
                +
                F.when(
                    F.col("pending_payment_count") > 0,
                    10,
                ).otherwise(0)
                +
                F.when(
                    F.col("paused_subscription_count") > 0,
                    10,
                ).otherwise(0)
                +
                F.when(
                    F.col("active_subscription_count") == 0,
                    15,
                ).otherwise(0)
            ),
        ),
    )
    .withColumn(
        "risk_tier",
        F.when(
            F.col("risk_score") >= 60,
            "High",
        )
        .when(
            F.col("risk_score") >= 30,
            "Medium",
        )
        .otherwise("Low"),
    )
    .withColumn(
        "customer_value_tier",
        F.when(
            (F.col("customer_segment") == "VIP")
            | (F.col("current_mrr") >= 250),
            "High Value",
        )
        .when(
            (F.col("customer_segment") == "Premium")
            | (F.col("current_mrr") >= 100),
            "Growth Value",
        )
        .otherwise("Standard Value"),
    )
)


# Create the final Customer 360 DataFrame.

customer_360_df = (
    customer_360_scored_df
    .withColumn(
        "_gold_generated_at",
        F.current_timestamp(),
    )
)

customer_360_count = customer_360_df.count()

distinct_customer_360_count = (
    customer_360_df
    .select("customer_id")
    .distinct()
    .count()
)

duplicate_customer_360_count = (
    customer_360_count
    - distinct_customer_360_count
)

assert customer_360_count == EXPECTED_CUSTOMER_COUNT, (
    "Unexpected Customer 360 record count. "
    f"Expected {EXPECTED_CUSTOMER_COUNT:,}, "
    f"received {customer_360_count:,}."
)

assert duplicate_customer_360_count == 0, (
    "Duplicate Customer 360 records detected."
)

print(
    "Customer 360 records: "
    f"{customer_360_count:,}"
)

print(
    "Distinct Customer 360 IDs: "
    f"{distinct_customer_360_count:,}"
)

print(
    "Analytics snapshot date: "
    f"{analytics_snapshot_date}"
)

display(
    customer_360_df
    .groupBy(
        "risk_tier",
        "customer_value_tier",
    )
    .agg(
        F.count("*").alias("customer_count"),
        F.round(
            F.sum("current_mrr"),
            2,
        ).alias("current_mrr"),
        F.round(
            F.sum("outstanding_amount"),
            2,
        ).alias("outstanding_amount"),
        F.round(
            F.sum("recovered_retry_amount"),
            2,
        ).alias("recovered_retry_amount"),
    )
    .orderBy(
        "risk_tier",
        "customer_value_tier",
    )
)

## 6. Validate and Reconcile Customer 360

Validate Customer 360 uniqueness, metric relationships, risk classifications, and financial totals.

Customer-level aggregates must reconcile exactly with the Silver subscription, invoice, and payment tables. A deterministic record hash is generated from all analytical attributes to support idempotent Gold merges.

In [0]:
gold_hash_columns = [
    column_name
    for column_name in customer_360_df.columns
    if column_name != "_gold_generated_at"
]

customer_360_df = (
    customer_360_df
    .withColumn(
        "_gold_record_hash",
        F.sha2(
            F.to_json(
                F.struct(
                    *[
                        F.col(column_name)
                        for column_name
                        in gold_hash_columns
                    ]
                ),
                options={
                    "ignoreNullFields": "false"
                },
            ),
            256,
        ),
    )
)

critical_customer_360_columns = [
    "customer_id",
    "customer_name",
    "email",
    "country",
    "region",
    "customer_segment",
    "customer_value_tier",
    "signup_date",
    "customer_status",
    "risk_score",
    "risk_tier",
    "analytics_snapshot_date",
    "_gold_record_hash",
]

critical_field_is_missing = None

for column_name in (
    critical_customer_360_columns
):
    missing_condition = (
        F.col(column_name).isNull()
        | (
            F.trim(
                F.col(column_name).cast("string")
            )
            == ""
        )
    )

    critical_field_is_missing = (
        missing_condition
        if critical_field_is_missing is None
        else critical_field_is_missing
        | missing_condition
    )

validation_conditions = {
    "null_critical_field_count":
        critical_field_is_missing,

    "invalid_subscription_count":
        F.col("subscription_count")
        != (
            F.col("active_subscription_count")
            + F.col("paused_subscription_count")
            + F.col(
                "cancelled_subscription_count"
            )
        ),

    "invalid_invoice_count":
        F.col("invoice_count")
        != (
            F.col("paid_invoice_count")
            + F.col("open_invoice_count")
            + F.col("past_due_invoice_count")
            + F.col("voided_invoice_count")
        ),

    "invalid_payment_count":
        F.col("payment_attempt_count")
        != (
            F.col("successful_payment_count")
            + F.col("failed_payment_count")
            + F.col("pending_payment_count")
        ),

    "invalid_retry_count":
        F.col("retry_attempt_count")
        != (
            F.col("successful_retry_count")
            + F.col("failed_retry_count")
        ),

    "invalid_mrr_arr_count":
        F.abs(
            F.col("current_arr")
            - (
                F.col("current_mrr")
                * F.lit(12)
            )
        )
        > F.lit(0.01),

    "invalid_invoice_balance_count":
        (
            F.abs(
                F.col("gross_invoiced_amount")
                - (
                    F.col(
                        "collectible_invoice_amount"
                    )
                    + F.col("voided_amount")
                )
            )
            > F.lit(0.01)
        )
        | (
            F.abs(
                F.col(
                    "collectible_invoice_amount"
                )
                - (
                    F.col("collected_amount")
                    + F.col("outstanding_amount")
                )
            )
            > F.lit(0.01)
        ),

    "invalid_settlement_count":
        F.abs(
            F.col("settled_amount")
            - F.col("collected_amount")
        )
        > F.lit(0.01),

    "invalid_risk_amount_count":
        (
            F.abs(
                F.col("revenue_at_risk_amount")
                - F.col("outstanding_amount")
            )
            > F.lit(0.01)
        )
        | (
            F.abs(
                F.col(
                    "confirmed_leakage_amount"
                )
                - F.col("past_due_amount")
            )
            > F.lit(0.01)
        ),

    "invalid_rate_count":
        ~F.col(
            "collection_rate_pct"
        ).between(0, 100)
        | ~F.col(
            "payment_attempt_success_rate_pct"
        ).between(0, 100)
        | ~F.col(
            "retry_recovery_rate_pct"
        ).between(0, 100),

    "invalid_risk_score_count":
        ~F.col("risk_score").between(0, 100),

    "invalid_risk_tier_count":
        (
            (F.col("risk_score") >= 60)
            & (F.col("risk_tier") != "High")
        )
        | (
            F.col("risk_score").between(30, 59)
            & (F.col("risk_tier") != "Medium")
        )
        | (
            (F.col("risk_score") < 30)
            & (F.col("risk_tier") != "Low")
        ),

    "invalid_value_tier_count":
        ~F.col("customer_value_tier").isin(
            "High Value",
            "Growth Value",
            "Standard Value",
        ),

    "invalid_negative_metric_count":
        (
            (F.col("customer_tenure_days") < 0)
            | (F.col("current_mrr") < 0)
            | (F.col("current_arr") < 0)
            | (F.col("collected_amount") < 0)
            | (F.col("outstanding_amount") < 0)
            | (F.col("voided_amount") < 0)
            | (
                F.col("recovered_retry_amount")
                < 0
            )
        ),
}

customer_360_validation_metrics = (
    customer_360_df
    .agg(
        F.count("*").alias(
            "customer_360_count"
        ),
        F.countDistinct("customer_id").alias(
            "distinct_customer_count"
        ),
        *[
            F.sum(
                F.when(
                    condition,
                    1,
                ).otherwise(0)
            ).alias(metric_name)
            for metric_name, condition
            in validation_conditions.items()
        ],
    )
    .first()
    .asDict()
)

duplicate_customer_count = (
    customer_360_validation_metrics[
        "customer_360_count"
    ]
    - customer_360_validation_metrics[
        "distinct_customer_count"
    ]
)

silver_invoice_totals = (
    silver_invoices_df
    .agg(
        F.sum("invoice_total_amount").alias(
            "gross_invoiced_amount"
        ),
        F.sum("amount_paid").alias(
            "collected_amount"
        ),
        F.sum("outstanding_amount").alias(
            "outstanding_amount"
        ),
        F.sum("voided_amount").alias(
            "voided_amount"
        ),
    )
    .first()
    .asDict()
)

silver_subscription_totals = (
    silver_subscriptions_df
    .agg(
        F.sum(
            F.when(
                F.col("subscription_status")
                == "Active",
                F.col(
                    "contracted_monthly_price"
                ),
            ).otherwise(F.lit(0))
        ).alias("current_mrr")
    )
    .first()
    .asDict()
)

silver_payment_totals = (
    silver_payments_df
    .agg(
        F.sum("settled_amount").alias(
            "settled_amount"
        ),
        F.sum(
            F.when(
                (
                    F.col("attempt_number") == 2
                )
                & (
                    F.col("payment_status")
                    == "Succeeded"
                ),
                F.col("settled_amount"),
            ).otherwise(F.lit(0))
        ).alias("recovered_retry_amount"),
    )
    .first()
    .asDict()
)

gold_customer_totals = (
    customer_360_df
    .agg(
        F.sum("gross_invoiced_amount").alias(
            "gross_invoiced_amount"
        ),
        F.sum("collected_amount").alias(
            "collected_amount"
        ),
        F.sum("outstanding_amount").alias(
            "outstanding_amount"
        ),
        F.sum("voided_amount").alias(
            "voided_amount"
        ),
        F.sum("current_mrr").alias(
            "current_mrr"
        ),
        F.sum("settled_amount").alias(
            "settled_amount"
        ),
        F.sum("recovered_retry_amount").alias(
            "recovered_retry_amount"
        ),
    )
    .first()
    .asDict()
)

reconciliation_pairs = {
    "gross_invoiced_amount": (
        silver_invoice_totals[
            "gross_invoiced_amount"
        ],
        gold_customer_totals[
            "gross_invoiced_amount"
        ],
    ),
    "collected_amount": (
        silver_invoice_totals[
            "collected_amount"
        ],
        gold_customer_totals[
            "collected_amount"
        ],
    ),
    "outstanding_amount": (
        silver_invoice_totals[
            "outstanding_amount"
        ],
        gold_customer_totals[
            "outstanding_amount"
        ],
    ),
    "voided_amount": (
        silver_invoice_totals[
            "voided_amount"
        ],
        gold_customer_totals[
            "voided_amount"
        ],
    ),
    "current_mrr": (
        silver_subscription_totals[
            "current_mrr"
        ],
        gold_customer_totals[
            "current_mrr"
        ],
    ),
    "settled_amount": (
        silver_payment_totals[
            "settled_amount"
        ],
        gold_customer_totals[
            "settled_amount"
        ],
    ),
    "recovered_retry_amount": (
        silver_payment_totals[
            "recovered_retry_amount"
        ],
        gold_customer_totals[
            "recovered_retry_amount"
        ],
    ),
}

for metric_name, metric_value in (
    customer_360_validation_metrics.items()
):
    print(
        f"{metric_name}: "
        f"{metric_value:,}"
    )

print(
    f"duplicate_customer_count: "
    f"{duplicate_customer_count:,}"
)

assert (
    customer_360_validation_metrics[
        "customer_360_count"
    ]
    == EXPECTED_CUSTOMER_COUNT
), "Unexpected Customer 360 count."

assert duplicate_customer_count == 0, (
    "Duplicate Customer 360 records detected."
)

for metric_name in (
    validation_conditions.keys()
):
    assert (
        customer_360_validation_metrics[
            metric_name
        ]
        == 0
    ), f"Validation failed: {metric_name}"

for metric_name, (
    silver_value,
    gold_value,
) in reconciliation_pairs.items():
    silver_numeric_value = float(
        silver_value or 0
    )

    gold_numeric_value = float(
        gold_value or 0
    )

    difference = abs(
        silver_numeric_value
        - gold_numeric_value
    )

    print(
        f"{metric_name}: "
        f"Silver={silver_numeric_value:,.2f}, "
        f"Gold={gold_numeric_value:,.2f}, "
        f"Difference={difference:,.2f}"
    )

    assert difference <= 0.01, (
        f"Reconciliation failed: "
        f"{metric_name}"
    )

print(
    "Customer 360 validation and reconciliation "
    "completed successfully."
)

display(
    customer_360_df
    .groupBy("risk_tier")
    .agg(
        F.count("*").alias("customer_count"),
        F.round(
            F.sum("current_mrr"),
            2,
        ).alias("current_mrr"),
        F.round(
            F.sum("collected_amount"),
            2,
        ).alias("collected_amount"),
        F.round(
            F.sum("outstanding_amount"),
            2,
        ).alias("outstanding_amount"),
        F.round(
            F.sum("past_due_amount"),
            2,
        ).alias("past_due_amount"),
        F.round(
            F.sum("recovered_retry_amount"),
            2,
        ).alias("recovered_retry_amount"),
    )
    .orderBy("risk_tier")
)

## 7. Persist the Customer 360 Gold Table

Persist one analytics-ready record per current customer in a managed Delta table.

The initial execution creates the Gold table. Subsequent executions use a deterministic record hash and Delta `MERGE` to insert new customers, update changed customer profiles, and remove customers no longer present in the current Silver state.

In [0]:
spark.sql(
    "CREATE SCHEMA IF NOT EXISTS "
    "workspace.revenue_leakage_gold"
)

customer_360_df.createOrReplaceTempView(
    "customer_360_gold_updates"
)

if spark.catalog.tableExists(
    GOLD_CUSTOMER_360_TABLE
):
    spark.sql(
        f"""
        MERGE INTO {GOLD_CUSTOMER_360_TABLE} AS target
        USING customer_360_gold_updates AS source
            ON target.customer_id = source.customer_id

        WHEN MATCHED
            AND target._gold_record_hash
                <> source._gold_record_hash
            THEN UPDATE SET *

        WHEN NOT MATCHED
            THEN INSERT *

        WHEN NOT MATCHED BY SOURCE
            THEN DELETE
        """
    )

    write_method = "Delta MERGE"

else:
    (
        customer_360_df
        .write
        .format("delta")
        .mode("overwrite")
        .option(
            "overwriteSchema",
            "true",
        )
        .saveAsTable(
            GOLD_CUSTOMER_360_TABLE
        )
    )

    write_method = (
        "Initial Delta table creation"
    )


saved_customer_360_df = spark.table(
    GOLD_CUSTOMER_360_TABLE
)

saved_customer_360_count = (
    saved_customer_360_df.count()
)

saved_distinct_customer_count = (
    saved_customer_360_df
    .select("customer_id")
    .distinct()
    .count()
)

saved_duplicate_customer_count = (
    saved_customer_360_count
    - saved_distinct_customer_count
)

saved_null_customer_id_count = (
    saved_customer_360_df
    .filter(
        F.col("customer_id").isNull()
    )
    .count()
)

assert (
    saved_customer_360_count
    == EXPECTED_CUSTOMER_COUNT
), (
    "Unexpected saved Customer 360 count."
)

assert (
    saved_distinct_customer_count
    == EXPECTED_CUSTOMER_COUNT
), (
    "Unexpected saved distinct customer count."
)

assert saved_duplicate_customer_count == 0, (
    "Duplicate saved Customer 360 records detected."
)

assert saved_null_customer_id_count == 0, (
    "Saved Customer 360 records contain null customer IDs."
)

print(
    "Write method:: "
    f"{write_method}"
)

print(
    "Gold table: "
    f"{GOLD_CUSTOMER_360_TABLE}"
)

print(
    "Saved Customer 360 records: "
    f"{saved_customer_360_count:,}"
)

print(
    "Saved distinct customer IDs: "
    f"{saved_distinct_customer_count:,}"
)

display(
    saved_customer_360_df
    .groupBy("risk_tier")
    .agg(
        F.count("*").alias(
            "customer_count"
        ),
        F.round(
            F.sum("current_mrr"),
            2,
        ).alias(
            "current_mrr"
        ),
        F.round(
            F.sum("collected_amount"),
            2,
        ).alias(
            "collected_amount"
        ),
        F.round(
            F.sum("outstanding_amount"),
            2,
        ).alias(
            "outstanding_amount"
        ),
        F.round(
            F.sum("past_due_amount"),
            2,
        ).alias(
            "past_due_amount"
        ),
        F.round(
            F.sum("recovered_retry_amount"),
            2,
        ).alias(
            "recovered_retry_amount"
        ),
    )
    .orderBy("risk_tier")
)

## 8. Validate Idempotent Gold Reprocessing

Rerun the Customer 360 Delta merge using the same validated source dataset.

A successful idempotency test must preserve the table row count and produce zero inserts, updates, or deletes because no Silver business data has changed.

In [0]:
rows_before_idempotency_rerun = (
    spark.table(
        GOLD_CUSTOMER_360_TABLE
    )
    .count()
)

customer_360_df.createOrReplaceTempView(
    "customer_360_gold_updates"
)

spark.sql(
    f"""
    MERGE INTO {GOLD_CUSTOMER_360_TABLE} AS target
    USING customer_360_gold_updates AS source
        ON target.customer_id = source.customer_id

    WHEN MATCHED
        AND target._gold_record_hash
            <> source._gold_record_hash
        THEN UPDATE SET *

    WHEN NOT MATCHED
        THEN INSERT *

    WHEN NOT MATCHED BY SOURCE
        THEN DELETE
    """
)

customer_360_after_rerun_df = spark.table(
    GOLD_CUSTOMER_360_TABLE
)

rows_after_idempotency_rerun = (
    customer_360_after_rerun_df.count()
)

distinct_customers_after_rerun = (
    customer_360_after_rerun_df
    .select("customer_id")
    .distinct()
    .count()
)

duplicate_customers_after_rerun = (
    rows_after_idempotency_rerun
    - distinct_customers_after_rerun
)

latest_customer_360_history_df = (
    spark.sql(
        f"""
        DESCRIBE HISTORY
        {GOLD_CUSTOMER_360_TABLE}
        """
    )
    .orderBy(
        F.desc("version")
    )
    .limit(1)
)

latest_history_row = (
    latest_customer_360_history_df
    .first()
)

latest_operation_metrics = (
    latest_history_row[
        "operationMetrics"
    ]
    or {}
)

rows_inserted_during_rerun = int(
    latest_operation_metrics.get(
        "numTargetRowsInserted",
        0,
    )
)

rows_updated_during_rerun = int(
    latest_operation_metrics.get(
        "numTargetRowsUpdated",
        0,
    )
)

rows_deleted_during_rerun = int(
    latest_operation_metrics.get(
        "numTargetRowsDeleted",
        0,
    )
)

assert (
    rows_before_idempotency_rerun
    == EXPECTED_CUSTOMER_COUNT
), (
    "Unexpected row count before the idempotency rerun."
)

assert (
    rows_after_idempotency_rerun
    == EXPECTED_CUSTOMER_COUNT
), (
    "Unexpected row count after the idempotency rerun."
)

assert rows_inserted_during_rerun == 0, (
    "Customer 360 rerun inserted unexpected rows."
)

assert rows_updated_during_rerun == 0, (
    "Customer 360 rerun updated unexpected rows."
)

assert rows_deleted_during_rerun == 0, (
    "Customer 360 rerun deleted unexpected rows."
)

assert duplicate_customers_after_rerun == 0, (
    "Duplicate customers detected after the rerun."
)

print(
    "Rows before rerun: "
    f"{rows_before_idempotency_rerun:,}"
)

print(
    "Rows after rerun: "
    f"{rows_after_idempotency_rerun:,}"
)

print(
    "Rows inserted during rerun: "
    f"{rows_inserted_during_rerun:,}"
)

print(
    "Rows updated during rerun: "
    f"{rows_updated_during_rerun:,}"
)

print(
    "Rows deleted during rerun: "
    f"{rows_deleted_during_rerun:,}"
)

print(
    "Duplicate customers after rerun: "
    f"{duplicate_customers_after_rerun:,}"
)

print(
    "Customer 360 Gold processing is idempotent."
)

display(
    latest_customer_360_history_df
    .select(
        "version",
        "timestamp",
        "operation",
        "operationMetrics",
    )
)

## 9. Final Result

The Customer 360 Gold model was created successfully as an analytics-ready Delta table.

### Output

- **Gold table:** `workspace.revenue_leakage_gold.customer_360`
- **Customer records:** 5,150
- **Distinct customer IDs:** 5,150
- **High-risk customers:** 1,003
- **Medium-risk customers:** 1,503
- **Low-risk customers:** 2,644
- **Current MRR:** 450,435.70 USD
- **Current ARR:** 5,405,228.40 USD
- **Collected revenue:** 3,806,266.54 USD
- **Outstanding revenue:** 729,286.01 USD
- **Past-due revenue:** 646,033.53 USD
- **Recovered retry revenue:** 583,858.90 USD

### Quality Guarantees

- One record per current Silver customer
- Zero duplicate customer records
- Zero null critical fields
- Exact Silver-to-Gold financial reconciliation
- Valid subscription, invoice, payment, risk, and value metrics
- Deterministic Gold record hashes
- Idempotent Delta merge processing
- Zero inserts, updates, or deletes during unchanged reprocessing